<a href="https://colab.research.google.com/github/helbertvenegas/Property-PI-for-double-Ore-extension-of-type-14641/blob/main/calculos_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cuaderno PBW
### James Gómez y Helbert Venegas

**Versión interactiva para Google Colab y Jupyter.** En Colab, pulsa **Entorno de ejecución → Ejecutar todas**. Al final, en el **Panel interactivo**, elige una familia o modelo, cambia los parámetros y pulsa **Calcular**. Puedes añadir varios casos al comparador y exportarlos a CSV.

En GitHub se ve el catálogo y los ejemplos guardados. Los menús y botones se activan al ejecutar el cuaderno en Colab o Jupyter.

| Contenido | Alcance |
|---|---|
| Catálogo A–Z | Las 26 familias de tipo (14641), criterios PI y ejemplos editables |
| Módulos especiales | G, Heisenberg, Bell–Smith, matrices cuánticas estándar y de Takeuchi |
| Herramientas | Smith, Pfaffiano, centralidad monomial y polinomios de Dickson |

El cuaderno usa aritmética exacta. Escribe **zeta(N,r)** para $\exp(2\pi i r/N)$, **I** para $\sqrt{-1}$ y, por ejemplo, **1/2** o **sqrt(3)**. Los grados abiertos y los invariantes documentados solo en casos especiales se distinguen de los valores calculables.

[Cuaderno PBW en ChatGPT Sites](https://cuaderno-pbw.foxyllama9.chatgpt.site)

## Catálogo visible de las 26 familias

La tabla resume el criterio PI del manuscrito bajo sus hipótesis de admisibilidad. Aquí raíz significa raíz de la unidad y s²=f.

| Familia | Criterio PI |
|:---:|---|
| **A** | Nunca |
| **B** | h raíz de la unidad |
| **C** | 3h² raíz de la unidad |
| **D** | p,h raíces de la unidad y p≠±1 |
| **E** | 2h² raíz de la unidad |
| **F** | 2h² raíz de la unidad |
| **G** | p,h raíces de la unidad |
| **H** | Nunca |
| **I** | 2h² raíz de la unidad |
| **J** | 2h² raíz de la unidad |
| **K** | q,f,h raíces de la unidad |
| **L** | q,f,h raíces de la unidad |
| **M** | f≠0 y h(s+1), h(s−1) raíces de la unidad |
| **N** | f+g, f−g raíces de la unidad |
| **O** | f≠0 y h(1+s), h(1−s) raíces de la unidad |
| **P** | f≠0 y h(1+s), h(1−s) raíces de la unidad |
| **Q** | h raíz de la unidad |
| **R** | h raíz de la unidad |
| **S** | 2h raíz de la unidad |
| **T** | 2h raíz de la unidad |
| **U** | 2h raíz de la unidad |
| **V** | h raíz de la unidad |
| **W** | f≠0 y h(s+i), h(s−i) raíces de la unidad |
| **X** | Nunca |
| **Y** | Nunca |
| **Z** | h(s+i), h(s−i) raíces de la unidad |

El manuscrito da cotas para C, F e I. Los grados PI de Q, R y V se indican como abiertos en esa versión.

In [1]:
# @title 00 · Preparar funciones
from csv import DictWriter
from functools import lru_cache
from math import gcd, isqrt, lcm, prod
from pathlib import Path
from string import ascii_uppercase

import sympy as sp
from sympy.matrices.normalforms import smith_normal_form


def entero(x, nombre):
    if isinstance(x, bool) or not isinstance(x, (int, sp.Integer)):
        raise ValueError(f"{nombre} debe ser un entero.")
    return int(x)


def positivo(x, nombre):
    x = entero(x, nombre)
    if x < 1:
        raise ValueError(f"{nombre} debe ser positivo.")
    return x


def zeta(n, r=1):
    n = positivo(n, "n")
    return sp.exp(2 * sp.pi * sp.I * sp.Rational(entero(r, "r") % n, n))


def escalar(x):
    if isinstance(x, (str, bool)):
        raise ValueError("Usa un entero, sp.Rational, sp.sqrt o zeta.")
    x = sp.sympify(x)
    if not isinstance(x, sp.Expr) or x.has(sp.Float) or x.free_symbols:
        raise ValueError("Se requiere un escalar algebraico exacto, sin símbolos ni decimales.")
    return x


T = sp.Symbol("T")


def polinomio_minimo(x):
    return sp.Poly(sp.minimal_polynomial(escalar(x), T), T, domain=sp.QQ).monic()


def igual(a, b):
    dif = sp.simplify(escalar(a) - escalar(b))
    return dif == 0 or polinomio_minimo(dif).as_expr() == T


@lru_cache(maxsize=128)
def candidatos_ciclotomicos(grado):
    # φ(n)^2 ≥ n/2, por factorización en potencias primas.
    return [n for n in range(1, 2 * grado * grado + 1)
            if sp.totient(n) == grado]


@lru_cache(maxsize=256)
def orden(x):
    # Orden multiplicativo finito, o None si x no es raíz de la unidad.
    pol = polinomio_minimo(x)
    if not pol.is_cyclotomic:
        return None
    for n in candidatos_ciclotomicos(pol.degree()):
        if pol.as_expr() == sp.cyclotomic_poly(n, T):
            return n
    raise ArithmeticError("No se pudo identificar el polinomio ciclotómico.")


def fase(x, n):
    # Devuelve r tal que x = ζ_n^r para una raíz de orden que divide n.
    if igual(x, 1):
        return 0
    m = orden(x)
    if m is None or n % m:
        raise ValueError("El orden del escalar debe dividir el módulo.")
    for k in range(1, m):
        if gcd(k, m) == 1 and igual(x, zeta(m, k)):
            return (n // m) * k
    raise ArithmeticError("No se pudo hallar la fase.")


def ficha(datos):
    for campo, valor in datos.items():
        if isinstance(valor, sp.MatrixBase):
            print(f"{campo}:")
            sp.pprint(valor)
        else:
            print(f"{campo}: {valor}")

## 1. Retículo central de un toro cuántico

Usamos $u_i u_j=\zeta_N^{B_{ij}}u_j u_i$, con $B$ alternante. El monomio de exponente $a$ es central cuando $Ba=0\bmod N$. Si $d_i$ son los factores no nulos de Smith, el índice central es $\prod_i N/\gcd(N,d_i)$. El grado PI es su raíz cuadrada. En la versión afín se restringe a exponentes no negativos.

In [2]:
# @title 01 · Matriz de conmutación
def alternante(dim, superiores):
    valores = list(superiores)
    if len(valores) != dim * (dim - 1) // 2:
        raise ValueError("Cantidad incorrecta de entradas en el triángulo superior.")
    B = sp.zeros(dim)
    entradas = iter(valores)
    for i in range(dim):
        for j in range(i + 1, dim):
            B[i, j] = entero(next(entradas), "Entrada de B")
            B[j, i] = -B[i, j]
    return B


def smith(B):
    if not B.rows or not B.cols:
        return ()
    return tuple(abs(int(v)) for v in smith_normal_form(B, domain=sp.ZZ).diagonal() if v)


def tamano_imagen(B, N):
    return prod(N // gcd(N, d) for d in smith(B))


def pfaffiano(B):
    if B.rows % 2:
        return None
    if B.rows == 0:
        return 1
    suma = 0
    for j in range(1, B.rows):
        if B[0, j]:
            indices = [k for k in range(1, B.rows) if k != j]
            suma += (-1)**(j + 1) * B[0, j] * pfaffiano(B.extract(indices, indices))
    return suma


def toro(B, N, exponente=None):
    B, N = sp.Matrix(B), positivo(N, "N")
    if B.rows == 0 or B.rows != B.cols or B.T != -B or not all(v.is_Integer for v in B):
        raise ValueError("B debe ser una matriz cuadrada alternante de enteros.")
    factores = smith(B)
    indice = prod(N // gcd(N, d) for d in factores)
    grado = isqrt(indice)
    if grado**2 != indice:
        raise ArithmeticError("El índice debe ser un cuadrado.")
    pf = pfaffiano(B) if B.rows <= 12 else "Omitido para dimensión mayor que 12"
    resultado = {"Módulo": N, "Smith": factores, "Índice del retículo": indice,
                 "Grado PI": grado, "Pfaffiano": pf, "Matriz": B}
    if exponente is not None:
        if len(exponente) != B.cols:
            raise ValueError("El exponente debe tener una entrada por generador.")
        v = sp.Matrix([entero(a, "Exponente") for a in exponente])
        residuo = tuple(int(x) % N for x in B * v)
        resultado.update({"Residuo": residuo, "Monomio central": not any(residuo)})
    return resultado


def orden_ozono_diagonal(B, C, N):
    # Orden de im(B mod N) ∩ ker(C mod N).
    return tamano_imagen(B, N) // tamano_imagen(C * B, N)


def peso_fijo(B, C, N, exponente):
    # Comprueba si a·e=0 para todo e en im(B) ∩ ker(C), módulo N.
    if len(exponente) != B.rows:
        raise ValueError("Se requiere una entrada por generador.")
    a = sp.Matrix([[entero(e, "Exponente") for e in exponente]])
    R = C * B
    return tamano_imagen(R.col_join(a * B), N) == tamano_imagen(R, N)


B = alternante(4, [1, 0, 0, 0, 0, 2])
ficha(toro(B, 6, exponente=(6, 0, 0, 0)))

Módulo: 6
Smith: (1, 1, 2, 2)
Índice del retículo: 324
Grado PI: 18
Pfaffiano: 2
Matriz:
⎡0   1  0   0⎤
⎢            ⎥
⎢-1  0  0   0⎥
⎢            ⎥
⎢0   0  0   2⎥
⎢            ⎥
⎣0   0  -2  0⎦
Residuo: (0, 0, 0, 0)
Monomio central: True


## 2. Familia $G(p,f,h)$

Introduce $p=\zeta_N^r$ y $h=\zeta_N^s$. El cuaderno reduce el módulo al orden de $\langle p,h\rangle$ y comprueba el grado PI por Smith y por la fórmula cerrada. La matriz birracional usa $(x_1,T,y_1,y_2)$ y el cálculo de ozono usa la matriz del graduado en $(x_1,x_2,y_1,y_2)$. Para $h=1$ y $h=p^{-1}$ se registran los generadores documentados del centro afín.

In [3]:
# @title 02 · Familia G
def familia_g(N, r, s, f=1, exponente=None):
    N, r, s = positivo(N, "N"), entero(r, "r"), entero(s, "s")
    if igual(f, 0):
        raise ValueError("La familia G exige f ≠ 0.")
    divisor = gcd(N, r, s)
    N = N // divisor
    r, s = (r // divisor) % N, (s // divisor) % N
    n = N // gcd(N, r)
    if n < 3:
        raise ValueError("La familia G exige p ≠ ±1.")
    B = alternante(4, [0, -(r+s), -(r+s), -r, r, -r])
    A = alternante(4, [0, -(r+s), -(r+s), -(2*r+s), -s, -r])
    info = toro(B, N)
    if info["Grado PI"] != N**2 // gcd(N, 2*r*(r+s)):
        raise ArithmeticError("La forma de Smith discrepa de la fórmula para G.")
    C = sp.Matrix([[-1, 1, 1, -1], [-2, 2, 0, 0]])
    if s == 0:
        centro = ("k[x₁ⁿ, y₁ⁿ, y₂ⁿ, Dₙ(x₂,αx₁²)]" if n % 2 else
                  "k[X,Y₁,Y₂,P,Υ]/(Υ²−(−1)ᵐXY₁Y₂), m=n/2, P=Dₘ(x₂,αx₁²)")
    elif (s + r) % N == 0:
        centro = "k[x₁,Y₁,Y₂,C,U] con relación según la paridad de n; U=Dₙ(x₂,αx₁²)"
    else:
        centro = "Sin presentación afín para este giro en la ficha"
    info.update({"Caso": "G", "PI": True, "ord(p)": n, "r": r, "s": s,
                 "Orden de Oz": orden_ozono_diagonal(A, C, N),
                 "Ozono": "e ∈ im(A mod N), e₂−e₁=e₄−e₃ y 2(e₂−e₁)=0",
                 "Matriz del graduado A": A,
                 "Centro de fracciones": "Monomios Laurent con exponente en ker(B mod N)",
                 "Centro afín": centro, "Parámetro Dickson α": "−f/(p²−1)²",
                 "Nakayama": "diag((hp)⁻²,(hp)⁻²,(hp)²,(hp)²)",
                 "Calabi–Yau": (2*(r+s)) % N == 0})
    if exponente is not None:
        if any(entero(e, "Exponente") < 0 for e in exponente):
            raise ValueError("El monomio PBW afín requiere exponentes no negativos.")
        info["Monomio fijo por Oz"] = peso_fijo(A, C, N, exponente)
    return info


g_ejemplo = familia_g(N=6, r=1, s=-1, f=1, exponente=(0, 2, 0, 0))
ficha(g_ejemplo)

Módulo: 6
Smith: (1, 1, 12, 12)
Índice del retículo: 36
Grado PI: 6
Pfaffiano: 12
Matriz:
⎡0  0   -6  -6⎤
⎢             ⎥
⎢0  0   -1  1 ⎥
⎢             ⎥
⎢6  1   0   -1⎥
⎢             ⎥
⎣6  -1  1   0 ⎦
Caso: G
PI: True
ord(p): 6
r: 1
s: 5
Orden de Oz: 4
Ozono: e ∈ im(A mod N), e₂−e₁=e₄−e₃ y 2(e₂−e₁)=0
Matriz del graduado A:
⎡0  0  -6  -6⎤
⎢            ⎥
⎢0  0  -7  -5⎥
⎢            ⎥
⎢6  7  0   -1⎥
⎢            ⎥
⎣6  5  1   0 ⎦
Centro de fracciones: Monomios Laurent con exponente en ker(B mod N)
Centro afín: k[x₁,Y₁,Y₂,C,U] con relación según la paridad de n; U=Dₙ(x₂,αx₁²)
Parámetro Dickson α: −f/(p²−1)²
Nakayama: diag((hp)⁻²,(hp)⁻²,(hp)²,(hp)²)
Calabi–Yau: True
Monomio fijo por Oz: True


## 3. Heisenberg cuántico de dos parámetros

Usamos $zx=p^{-1}xz$, $zy=pyz$, $yx=qxy-z$. Para $pq\ne1$, se define $\Omega=(yx-p^{-1}xy)/(q-p^{-1})$ y $\sigma(z)=pz$, $\sigma(\Omega)=q^{-1}\Omega$.

Si $\ell=|\langle p,q\rangle|$, el centro es $R[U,V]/(UV-F)$, con $R=k[z,\Omega]^{\langle\sigma\rangle}$, $U=x^\ell$, $V=y^\ell$ y $F=\prod_{j=0}^{\ell-1}\sigma^j(yx)$. Para el anillo fijo por el ozono, se sustituye $R$ por $k[z,\Omega]^{\langle\sigma^t\rangle}$, con $t=\operatorname{ord}(pq)$.

[Prepublicación de Gómez, Suárez y Venegas](https://arxiv.org/abs/2610.31781)

In [4]:
# @title 03 · Heisenberg
def heisenberg(N, r, s):
    N, r, s = positivo(N, "N"), entero(r, "r"), entero(s, "s")
    d = gcd(N, r, s)
    ell = N // d
    r, s = (r // d) % ell, (s // d) % ell
    t = ell // gcd(ell, r+s)
    if t == 1:
        return {"Caso": "Heisenberg", "PI": False,
                "Motivo": "pq=1. La acción de Weyl tiene orden infinito."}
    e = ell // t
    resultado = toro(alternante(3, [0, -r, s]), ell)
    resultado.update({"Caso": "Heisenberg", "PI": True, "ℓ": ell, "t": t,
                      "Ozono": f"μ_{ell} × μ_{e}", "Orden de Oz": ell*e,
                      "Exponente zⁱΩʲ para el centro": f"{r}i−{s}j ≡ 0 (mod {ell})",
                      "Exponente zⁱΩʲ para el anillo fijo": f"{r}i−{s}j ≡ 0 (mod {e})",
                      "Anillo fijo de intersección completa": e in (1, 2)})
    return resultado


h_ejemplo = heisenberg(N=5, r=1, s=2)
ficha(h_ejemplo)

Módulo: 5
Smith: (1, 1)
Índice del retículo: 25
Grado PI: 5
Pfaffiano: None
Matriz:
⎡0  0   -1⎤
⎢         ⎥
⎢0  0   2 ⎥
⎢         ⎥
⎣1  -2  0 ⎦
Caso: Heisenberg
PI: True
ℓ: 5
t: 5
Ozono: μ_5 × μ_1
Orden de Oz: 5
Exponente zⁱΩʲ para el centro: 1i−2j ≡ 0 (mod 5)
Exponente zⁱΩʲ para el anillo fijo: 1i−2j ≡ 0 (mod 1)
Anillo fijo de intersección completa: True


## 4. Bell–Smith normalizado

Relaciones $yz-\alpha zy=\lambda_0+\lambda_1x$, $zx-\beta xz=\mu_0+\mu_2y$ y $xy-\gamma yx=\nu_0+\nu_3z$. Introduce los exponentes de $\alpha,\beta,\gamma$ en una raíz común $\zeta_N$. En **activos** se declaran los coeficientes no nulos. Se verifica la condición PBW de la forma normalizada y se comprueba si algún giro activo vale $1$.

In [5]:
# @title 04 · Bell–Smith
def bell_smith(N, a, b, g, activos=(), exponente=None):
    N = positivo(N, "N")
    a, b, g = (entero(x, "Exponente del parámetro") % N for x in (a, b, g))
    datos = {"l0": (a, (0,1,1), b == g), "l1": (a+g, (-1,1,1), b == g),
             "m0": (b, (1,0,1), a == g), "m2": (b+a, (1,-1,1), a == g),
             "v0": (g, (1,1,0), a == b), "v3": (g+b, (1,1,-1), a == b)}
    activos = tuple(dict.fromkeys(activos))
    if any(k not in datos for k in activos):
        raise ValueError("Activos válidos: l0, l1, m0, m2, v0, v3.")
    if any(not datos[k][2] for k in activos):
        raise ValueError("El soporte incumple la condición PBW normalizada.")
    giros = {k: N // gcd(N, datos[k][0]) for k in activos}
    if 1 in giros.values():
        return {"Caso": "Bell–Smith", "PI": False, "Órdenes de giros": giros}
    B = alternante(3, [g, -b, a])
    C = sp.Matrix([datos[k][1] for k in activos]) if activos else sp.zeros(0, 3)
    resultado = toro(B, N)
    resultado.update({"Caso": "Bell–Smith", "PI": True, "Órdenes de giros": giros,
                      "Orden de Oz": orden_ozono_diagonal(B, C, N),
                      "Centro graduado": "gr Z(A) = Z(gr A) según el manuscrito",
                      "Restricciones en los pesos de Oz": C})
    if activos and set(activos) <= {"v0", "v3"}:
        resultado["Defecto δ"] = lcm(*giros.values())
    if exponente is not None:
        resultado["Monomio fijo por Oz"] = peso_fijo(B, C, N, exponente)
    return resultado


ficha(bell_smith(5, 1, 1, 2, activos=("v3",), exponente=(1, 1, 1)))

Módulo: 5
Smith: (1, 1)
Índice del retículo: 25
Grado PI: 5
Pfaffiano: None
Matriz:
⎡0   2   -1⎤
⎢          ⎥
⎢-2  0   1 ⎥
⎢          ⎥
⎣1   -1  0 ⎦
Caso: Bell–Smith
PI: True
Órdenes de giros: {'v3': 5}
Orden de Oz: 5
Centro graduado: gr Z(A) = Z(gr A) según el manuscrito
Restricciones en los pesos de Oz:
[1  1  -1]
Defecto δ: 5
Monomio fijo por Oz: True


## 5. Matrices cuánticas estándar $\mathcal O_q(M_n)$

Para $m=\operatorname{ord}(q)\ge3$, se calcula el grado PI en la presentación cuasipolinomial, en orden lexicográfico por filas. La entrada superior es $1$ para pares en una misma fila o columna y $0$ para los demás. Para $m$ par y $n>2$, se indica el rango de ozono documentado en el manuscrito.

[Jakobsen y Zhang, *The Center of the Quantized Matrix Algebra*](https://doi.org/10.1006/jabr.1997.7121)

In [6]:
# @title 05 · Matrices cuánticas
def matrices_cuanticas(n, m):
    n, m = positivo(n, "n"), positivo(m, "m")
    if n < 2 or m < 3:
        raise ValueError("Este módulo usa n ≥ 2 y m ≥ 3.")
    lugares = [(i, j) for i in range(n) for j in range(n)]
    B = alternante(n*n, [int(i == r or j == s)
                          for k, (i, j) in enumerate(lugares) for r, s in lugares[k+1:]])
    resultado = toro(B, m)
    resultado["Caso"] = f"O_q(M_{n})"
    if m % 2:
        resultado.update({"Ozono": f"μ_{m}^{n-1}", "Orden de Oz": m**(n-1),
                          "Anillo fijo": "Sumas de filas y columnas reflejadas iguales módulo m"})
    elif n == 2:
        resultado.update({"Ozono": f"μ_{m} × μ_2", "Orden de Oz": 2*m})
    else:
        resultado.update({"Subgrupo de Oz": f"μ_{m}^{n-1} × μ_2",
                          "Órdenes posibles de Oz": tuple(2**r * m**(n-1) for r in range(1, n)),
                          "Estado de Oz": "Parcial"})
    return resultado


ficha(matrices_cuanticas(n=3, m=5))

Módulo: 5
Smith: (1, 1, 1, 1, 2, 2)
Índice del retículo: 15625
Grado PI: 125
Pfaffiano: None
Matriz:
⎡0   1   1   1   0   0   1   0   0⎤
⎢                                 ⎥
⎢-1  0   1   0   1   0   0   1   0⎥
⎢                                 ⎥
⎢-1  -1  0   0   0   1   0   0   1⎥
⎢                                 ⎥
⎢-1  0   0   0   1   1   1   0   0⎥
⎢                                 ⎥
⎢0   -1  0   -1  0   1   0   1   0⎥
⎢                                 ⎥
⎢0   0   -1  -1  -1  0   0   0   1⎥
⎢                                 ⎥
⎢-1  0   0   -1  0   0   0   1   1⎥
⎢                                 ⎥
⎢0   -1  0   0   -1  0   -1  0   1⎥
⎢                                 ⎥
⎣0   0   -1  0   0   -1  -1  -1  0⎦
Caso: O_q(M_3)
Ozono: μ_5^2
Orden de Oz: 25
Anillo fijo: Sumas de filas y columnas reflejadas iguales módulo m


## 6. Las 26 familias de tipo (14641)

La función aplica la tabla del manuscrito **Polynomial identities in the double extension regular algebras of type (14641)**, versión **Auditoría 2**. La letra y la convención de parámetros se refieren a la [clasificación de Zhang y Zhang](https://arxiv.org/abs/0712.2550). **Fórmula del manuscrito** identifica un resultado transcrito de ese texto. Para C, F e I se devuelven cotas y para Q, R y V se conserva el estado abierto que figura en esa versión.

Los órdenes se calculan mediante polinomios mínimos y ciclotómicos. Para un valor racional usa, por ejemplo, **sp.Rational(-1, 2)**. Para M, O, P, W y Z se puede suministrar **raiz_f** para elegir la raíz cuadrada de $f$. Para N se usan $f,g$ y no se usa el giro $h$.

In [7]:
# @title 06 · Evaluar las familias A–Z
def familia_14641(familia, *, p=None, q=None, h=1, f=None, g=None, raiz_f=None):
    k = familia.strip().upper()
    if len(k) != 1 or k not in ascii_uppercase:
        raise ValueError("Elige una letra de A a Z.")
    usados = {"h": h} if k != "N" else {}

    def exigir(nombre, valor, no_nulo=False):
        if valor is None:
            raise ValueError(f"La familia {k} requiere {nombre}.")
        valor = escalar(valor)
        if no_nulo and igual(valor, 0):
            raise ValueError(f"{nombre} debe ser distinto de cero.")
        usados[nombre] = valor
        return valor

    def salida(pi, grados=None, estado="Fórmula del manuscrito", motivo=""):
        return {"Familia": k, "Parámetros": ", ".join(f"{u}={v}" for u, v in usados.items()),
                "PI": pi, "Grado PI": grados, "Estado": estado, "Motivo": motivo}

    if k != "N":
        h = exigir("h", h, no_nulo=True)
    if k in "BCDEFG":
        p = exigir("p", p, no_nulo=True)
        if k in "BEF" and orden(p) != 4:
            raise ValueError(f"{k} exige p²=−1.")
        if k == "C" and orden(p) != 3:
            raise ValueError("C exige que p sea una raíz cúbica primitiva.")
    if k in "IJKL":
        q = exigir("q", q, no_nulo=True)
        if k in "IJ" and orden(q) != 4:
            raise ValueError(f"{k} exige q²=−1.")
    if k in "GKLMNOPWZ":
        f = exigir("f", f, no_nulo=k in "GKLZ")
    if k == "N":
        g = exigir("g", g)
        if igual(f**2, g**2):
            raise ValueError("N exige f²≠g².")
    if k in "MOP" and igual(f, 1) or k in "WZ" and igual(f, -1):
        raise ValueError(f"El valor de f no es admisible para {k}.")
    if k == "G" and orden(p) in (1, 2):
        raise ValueError("G exige p≠±1.")
    if k in "AHXY":
        return salida(False, estado="Criterio del manuscrito", motivo="Nunca PI en característica cero.")
    if k == "D" and orden(p) in (1, 2) or k in "MOPW" and igual(f, 0):
        return salida(False, estado="Criterio del manuscrito", motivo="Especialización no PI.")

    if k in "MOPWZ":
        s = sp.sqrt(f) if raiz_f is None else escalar(raiz_f)
        if not igual(s**2, f):
            raise ValueError("raiz_f debe satisfacer s²=f.")
        usados["s"] = s

    if k in "BQRV":
        pruebas = [h]
    elif k in "DG":
        pruebas = [p, h]
    elif k == "C":
        pruebas = [3*h**2]
    elif k in "EFIJ":
        pruebas = [2*h**2]
    elif k in "KL":
        pruebas = [q, f, h]
    elif k in "MOP":
        pruebas = [h*(1+s), h*(1-s)]
    elif k == "N":
        pruebas = [f+g, f-g]
    elif k in "STU":
        pruebas = [2*h]
    else:
        pruebas = [h*(s+sp.I), h*(s-sp.I)]
    if any(orden(x) is None for x in pruebas):
        return salida(False, estado="Criterio del manuscrito",
                      motivo="Algún escalar del criterio tiene orden infinito.")

    if k == "B":
        grado = 8 * orden(h**4)
    elif k in "DG":
        N = lcm(orden(p), orden(h), 2 if k == "D" else 1)
        r, w = fase(p, N), fase(h, N)
        grado = N**2 // gcd(N, 2*r*(r+w))
    elif k in "KL":
        N0 = lcm(2, orden(q), orden(f))
        N = lcm(N0, orden(h))
        grado = 2*N if k == "K" or (N // N0) % 2 else N
    elif k in "STU":
        grado = 8 * orden(16*h**4)
    elif k in "EJ":
        grado = 8 * orden(4*h**4)
    elif k in "CFI":
        base = lcm(3, orden(3*(2*p+1)*h**3)) if k == "C" else lcm(4, orden(2*h**2))
        return salida(True, (base, (9 if k == "C" else 4)*base), "Cotas inferior y superior")
    elif k in "QRV":
        return salida(True, estado="Grado PI abierto en el manuscrito")
    else:
        if k == "M":
            u, v = (s+1)/(s-1), h**2*(f-1)
        elif k == "N":
            u, v = (f-g)/(f+g), f**2-g**2
        elif k in "OP":
            u, v = (1+s)/(1-s), h**2*(1-f)
        else:
            u, v = (s-sp.I)/(s+sp.I), h**2*(f+1)
            if k == "Z":
                u = -u
        ou, ov = orden(u), orden(v)
        factor = 2 if k in "OWZ" and ou % 2 and ov % 2 else 4
        grado = factor * (ou // gcd(ou, 2)) * (ov // gcd(ov, 2))
    return salida(True, grado)


comparacion = [
    familia_14641("G", p=zeta(6), h=zeta(6, -1), f=1),
    familia_14641("F", p=sp.I, h=1/sp.sqrt(2)),
    familia_14641("Q", h=1),
    familia_14641("W", f=3, h=sp.Rational(-1, 2)),
    familia_14641("Z", f=3, h=sp.Rational(-1, 2)),
]

for caso in comparacion:
    print(f'{caso["Familia"]} · PI={caso["PI"]} · grado={caso["Grado PI"]} · {caso["Estado"]}')

G · PI=True · grado=6 · Fórmula del manuscrito
F · PI=True · grado=(4, 16) · Cotas inferior y superior
Q · PI=True · grado=None · Grado PI abierto en el manuscrito
W · PI=True · grado=12 · Fórmula del manuscrito
Z · PI=True · grado=6 · Fórmula del manuscrito


## 7. Polinomios de Dickson y exportación

La primera especie se calcula con $D_0=2$, $D_1=X$ y $D_j=X D_{j-1}-A D_{j-2}$. Para el centro de G se usa $X=x_2$ y $A=\alpha x_1^2$, con $\alpha=-f/(p^2-1)^2$.

La última celda guarda el CSV de los casos definidos en **comparacion**. En Colab puedes descargarlo desde el panel **Archivos**.

In [8]:
# @title 07 · Dickson y tabla CSV
def dickson(n, X, A):
    n = entero(n, "n")
    if n < 0:
        raise ValueError("n debe ser no negativo.")
    if n == 0:
        return sp.Integer(2)
    anterior, actual = sp.Integer(2), sp.sympify(X)
    for _ in range(2, n+1):
        anterior, actual = actual, sp.expand(X*actual-A*anterior)
    return actual


X, A = sp.symbols("X A")
print("D₆(X,A) =", dickson(6, X, A))

destino = Path("resultados_pbw.csv")
with destino.open("w", encoding="utf-8-sig", newline="") as archivo:
    escritor = DictWriter(archivo, fieldnames=list(comparacion[0]))
    escritor.writeheader()
    escritor.writerows(comparacion)
print("Tabla creada:", destino.name)

D₆(X,A) = -2*A**3 + 9*A**2*X**2 - 6*A*X**4 + X**6
Tabla creada: resultados_pbw.csv


## Referencias y alcance

- J. Zhang y J. J. Zhang, [*Double Extension Regular Algebras of Type (14641)*](https://arxiv.org/abs/0712.2550).
- J. Gómez y H. Venegas, *Polynomial identities in the double extension regular algebras of type (14641)*, manuscrito de trabajo, Auditoría 2. Tabla de criterios y grados.
- Manuscritos de trabajo del Cuaderno PBW sobre G, Bell–Smith y matrices cuánticas. En G se distinguen el centro afín, el centro de fracciones y el graduado asociado.
- J. Gómez, H. Suárez y H. Venegas, [*Ozone groups and centers of two-parameter quantum Heisenberg algebras*](https://arxiv.org/abs/2610.31781).

Los cálculos modulares verifican los invariantes del modelo indicado. La transferencia a cada álgebra deformada usa el resultado matemático correspondiente y sus hipótesis.

## Panel interactivo

Ejecuta las celdas anteriores y luego **Abrir panel interactivo**. El selector contiene las familias **A–Z** y siete módulos. Los parámetros vienen con un ejemplo editable. Usa **Añadir al comparador** para conservar el cálculo de la sesión y **Exportar CSV** para descargar la tabla. Si usas Jupyter local, el entorno debe tener SymPy e ipywidgets.

In [9]:
# @title Catálogo, parámetros y modelos del panel
import ast
import html
import re

# Cada opción muestra la condición del manuscrito y un caso editable.
CATALOGO = {
    "A": ("Nunca", {}),
    "B": ("h raíz de la unidad", {"p":"I", "h":"1"}),
    "C": ("3h² raíz de la unidad", {"p":"zeta(3)", "h":"1/sqrt(3)"}),
    "D": ("p,h raíces de la unidad y p≠±1", {"p":"zeta(5)", "h":"1"}),
    "E": ("2h² raíz de la unidad", {"p":"I", "h":"1/sqrt(2)"}),
    "F": ("2h² raíz de la unidad", {"p":"I", "h":"1/sqrt(2)"}),
    "G": ("p,h raíces de la unidad", {"p":"zeta(6)", "h":"zeta(6,-1)", "f":"1"}),
    "H": ("Nunca", {}),
    "I": ("2h² raíz de la unidad", {"q":"I", "h":"1/sqrt(2)"}),
    "J": ("2h² raíz de la unidad", {"q":"I", "h":"1/sqrt(2)"}),
    "K": ("q,f,h raíces de la unidad", {"q":"1", "f":"1", "h":"1"}),
    "L": ("q,f,h raíces de la unidad", {"q":"1", "f":"-1", "h":"1"}),
    "M": ("f≠0 y h(s+1), h(s−1) raíces de la unidad", {"h":"1/2", "f":"-3"}),
    "N": ("f+g, f−g raíces de la unidad", {"f":"0", "g":"1"}),
    "O": ("f≠0 y h(1+s), h(1−s) raíces de la unidad", {"h":"1/2", "f":"-3"}),
    "P": ("f≠0 y h(1+s), h(1−s) raíces de la unidad", {"h":"1/2", "f":"-3"}),
    "Q": ("h raíz de la unidad", {"h":"1"}),
    "R": ("h raíz de la unidad", {"h":"1"}),
    "S": ("2h raíz de la unidad", {"h":"1/2"}),
    "T": ("2h raíz de la unidad", {"h":"1/2"}),
    "U": ("2h raíz de la unidad", {"h":"1/2"}),
    "V": ("h raíz de la unidad", {"h":"1"}),
    "W": ("f≠0 y h(s+i), h(s−i) raíces de la unidad", {"h":"-1/2", "f":"3"}),
    "X": ("Nunca", {}),
    "Y": ("Nunca", {}),
    "Z": ("h(s+i), h(s−i) raíces de la unidad", {"h":"-1/2", "f":"3"}),
}
MODELOS = {
    "g": ("Familia G · modelo birracional", {"N":"6", "r":"1", "s":"-1", "f":"1", "exponente":"0, 2, 0, 0"},
          "Usa p=ζ_N^r, h=ζ_N^s. El vector se refiere a la base afín (x₁,x₂,y₁,y₂)."),
    "heis": ("Heisenberg cuántico", {"N":"5", "r":"1", "s":"2"},
             "Usa p=ζ_N^r, q=ζ_N^s y las relaciones zx=p⁻¹xz, zy=pyz, yx=qxy−z."),
    "bell": ("Bell–Smith normalizado", {"N":"5", "a":"1", "b":"1", "g":"2", "activos":"v3", "exponente":"1, 1, 1"},
             "Usa α=ζ_N^a, β=ζ_N^b, γ=ζ_N^g. El soporte admite l0, l1, m0, m2, v0, v3 separados por comas."),
    "qm": ("Matrices cuánticas O_q(M_n)", {"n":"3", "m":"5"},
           "m es el orden de q y n es el tamaño. Para m par y n>2, el cálculo de ozono es parcial."),
    "qm2": ("Matrices de Takeuchi M₂(α,β)", {"N":"6", "r":"1", "s":"3"},
            "Usa α=ζ_N^r y β=ζ_N^s. Se distingue el toro birracional del centro afín."),
    "toro": ("Toro o espacio afín cuántico", {"N":"6", "matriz":"0 1 0 0\n-1 0 0 0\n0 0 0 2\n0 0 -2 0", "exponente":"6, 0, 0, 0"},
             "Escribe una fila por línea. Los exponentes negativos corresponden al toro localizado."),
    "dickson": ("Polinomios de Dickson", {"n":"6", "X":"3", "A":"2"},
                "Muestra el polinomio formal y su evaluación en X y A."),
}

def leer_expresion(texto):
    """Interpreta una gramática de expresiones algebraicas sin ejecutar código."""
    if len(texto) > 160:
        raise ValueError("La expresión debe tener como máximo 160 caracteres.")
    arbol = ast.parse(texto.strip().replace("^", "**"), mode="eval")
    operadores = {ast.Add: lambda a,b: a+b, ast.Sub: lambda a,b: a-b,
                  ast.Mult: lambda a,b: a*b, ast.Div: lambda a,b: a/b}

    def leer(nodo):
        if isinstance(nodo, ast.Constant) and type(nodo.value) is int:
            return sp.Integer(nodo.value)
        if isinstance(nodo, ast.Name) and nodo.id in {"I", "i"}:
            return sp.I
        if isinstance(nodo, ast.UnaryOp) and isinstance(nodo.op, (ast.UAdd, ast.USub)):
            valor = leer(nodo.operand)
            return -valor if isinstance(nodo.op, ast.USub) else valor
        if isinstance(nodo, ast.BinOp):
            a, b = leer(nodo.left), leer(nodo.right)
            if isinstance(nodo.op, ast.Pow):
                if not b.is_Integer or abs(b) > 64:
                    raise ValueError("Usa exponentes enteros entre −64 y 64.")
                return a**b
            op = operadores.get(type(nodo.op))
            if op:
                return op(a, b)
        if isinstance(nodo, ast.Call) and isinstance(nodo.func, ast.Name) and not nodo.keywords:
            args = [leer(a) for a in nodo.args]
            if nodo.func.id == "sqrt" and len(args) == 1:
                return sp.sqrt(args[0])
            if nodo.func.id == "zeta" and len(args) in (1, 2):
                return zeta(*args)
        raise ValueError("Usa enteros, fracciones, I, sqrt(...) o zeta(n,r).")

    valor = leer(arbol.body)
    if valor.has(sp.zoo, sp.oo, -sp.oo, sp.nan):
        raise ValueError("La expresión debe ser un número algebraico finito.")
    return valor

def matriz_takeuchi(N, r, s):
    N, r, s = positivo(N, "N"), entero(r, "r"), entero(s, "s")
    divisor = gcd(N, r, s)
    N = N // divisor
    r, s = (r // divisor) % N, (s // divisor) % N
    t1, t2 = N // gcd(N, r+s), N // gcd(N, r-s)
    if t1 == 1:
        B = alternante(4, [-r, -s, 0, r-s, -s, -r])
        return {"Caso": "Takeuchi · caso escalar", "Base": "X₁₁,X₁₂,X₂₁,X₂₂",
                **toro(B, N), "Ozono": "im(B mod N)",
                "Centro y anillo fijo": "Monomios con exponentes no negativos en ker(B mod N)"}
    B = alternante(4, [-r, -s, 0, r-s, r-s, s-r])
    info = {"Caso": "Takeuchi", "Base birracional": "X₁₁,X₁₂,X₂₁,D",
            **toro(B, N), "ord(αβ)": t1, "ord(αβ⁻¹)": t2,
            "Centro de fracciones": "Monomios Laurent de la base birracional con exponentes en ker(B mod N)"}
    if r == s:
        datos = matrices_cuanticas(2, N)
        info.update({k: datos[k] for k in ("Ozono", "Orden de Oz")})
    elif t1 == t2:
        info.update({"Ozono": f"μ_{N} × μ_{N} × μ_{t1}", "Orden de Oz": N*N*t1,
                     "Centro afín": "Generado por Xᵢⱼ^N, X₁₁^t X₂₂^t y X₁₂^t X₂₁^t, t=ord(αβ)"})
    else:
        info["Estado de Oz"] = "Caso de órdenes distintos sin fórmula en esta ficha"
    return info

def evaluar_panel(modo, campos):
    if modo in CATALOGO:
        args = {k:leer_expresion(v) for k,v in campos.items()}
        info = familia_14641(modo, **args)
        info["Criterio PI"] = CATALOGO[modo][0]
        info["Interpretación del grado"] = (
            "Fórmula evaluada" if isinstance(info["Grado PI"], int) else
            "Cota inferior y superior" if isinstance(info["Grado PI"], tuple) else
            "Grado abierto en el manuscrito" if info["PI"] else "No aplica")
        if modo == "G" and info["PI"]:
            N = lcm(orden(args["p"]), orden(args["h"]))
            detalle = familia_g(N, fase(args["p"],N), fase(args["h"],N), args["f"])
            info.update({k:v for k,v in detalle.items() if k not in info})
        if modo == "L" and all(igual(args[k],v) for k,v in {"q":1,"f":-1,"h":1}.items()):
            info["Grupo de ozono"] = "μ₂³"
            info["Centro afín"] = "k[a,b,X₁,X₂,z]/(z²+b(a²−4b)X₁X₂)"
            info["Anillo fijo"] = "k[a,b,T,X₁,X₂,U]/(T²+b(a²−4b), U²−X₁X₂)"
        if modo == "S" and igual(4*args["h"]**2,1):
            info["Grupo de ozono"] = "μ₂⁴"
            info["Centro afín"] = "k[a,c,b,d]"
            info["Anillo fijo"] = "k[a,c,Tₓ,b,d,Tᵧ]/(Tₓ²+c(a²−4c), Tᵧ²+d(b²−4d))"
        return info
    def i(k):
        return int(campos[k].strip())
    def vector():
        partes = re.split(r"[,\s]+", campos["exponente"].strip())
        return tuple(map(int, partes)) if campos["exponente"].strip() else None

    if modo == "g":
        return familia_g(i("N"),i("r"),i("s"),leer_expresion(campos["f"]),vector())
    if modo == "heis":
        return heisenberg(i("N"),i("r"),i("s"))
    if modo == "bell":
        activos = tuple(s.strip() for s in campos["activos"].split(",") if s.strip())
        return bell_smith(i("N"),i("a"),i("b"),i("g"),activos,vector())
    if modo == "qm":
        return matrices_cuanticas(i("n"),i("m"))
    if modo == "qm2":
        return matriz_takeuchi(i("N"),i("r"),i("s"))
    if modo == "toro":
        filas = [list(map(int,re.split(r"[,\s]+",r.strip()))) for r in campos["matriz"].splitlines() if r.strip()]
        return toro(filas,i("N"),vector())
    x,a = sp.symbols("X A")
    n = i("n")
    return {"Caso": f"D_{n}", "Polinomio": dickson(n,x,a),
            "Evaluación exacta": dickson(n,leer_expresion(campos["X"]),leer_expresion(campos["A"]))}


In [10]:
# @title Abrir panel interactivo
import ipywidgets as widgets
from IPython.display import FileLink, display

def fila_html(info):
    filas = []
    for k, v in info.items():
        if isinstance(v, sp.MatrixBase):
            representacion = "<pre>" + html.escape(sp.pretty(v)) + "</pre>"
        else:
            texto = "Sí" if v is True else "No" if v is False else "—" if v is None else str(v)
            representacion = html.escape(texto)
        filas.append("<tr><th scope='row' style='text-align:left;padding:6px 12px 6px 0'>"
                     + html.escape(k) + "</th><td style='padding:6px'>"+representacion+"</td></tr>")
    return "<div style='overflow-x:auto'><table style='width:100%'>"+"".join(filas)+"</table></div>"

def abrir_panel(mostrar=True):
    ancho = widgets.Layout(width="100%")
    selector = widgets.Dropdown(
        options=[("Familia "+k, k) for k in CATALOGO]+[(v[0], k) for k,v in MODELOS.items()],
        value="G", description="Álgebra", layout=ancho)
    ayuda = widgets.HTML(layout=ancho)
    formulario = widgets.VBox(layout=ancho)
    resultado = widgets.HTML(layout=ancho)
    comparador = widgets.HTML(layout=ancho)
    estado = widgets.HTML(layout=ancho)
    enlaces = widgets.Output(layout=ancho)
    calcular = widgets.Button(description="Calcular", button_style="success", icon="calculator")
    agregar = widgets.Button(description="Añadir al comparador", icon="plus", disabled=True,
                             layout=widgets.Layout(width="180px"))
    descargar = widgets.Button(description="Exportar CSV", icon="download", disabled=True)
    limpiar = widgets.Button(description="Limpiar tabla", icon="trash", disabled=True)
    campos = {}
    guardados = []
    actual = None

    def valores():
        return {k:w.value for k,w in campos.items()}

    def actualizar_tabla():
        if not guardados:
            comparador.value = "<p>Todavía no hay casos guardados.</p>"
        else:
            titulos = list(guardados[0])
            cabecera = "".join("<th scope='col'>"+html.escape(k)+"</th>" for k in titulos)
            filas = "".join("<tr>"+"".join("<td>"+html.escape(str(r[k]))+"</td>" for k in titulos)+"</tr>" for r in guardados)
            comparador.value = "<div style='overflow-x:auto'><table><thead><tr>"+cabecera+"</tr></thead><tbody>"+filas+"</tbody></table></div>"
        descargar.disabled = limpiar.disabled = not guardados

    def marcar_pendiente(_=None):
        nonlocal actual
        actual = None
        agregar.disabled = True
        resultado.value = "<p>Parámetros listos. Pulsa <strong>Calcular</strong>.</p>"

    def configurar(_=None):
        modo = selector.value
        if modo in CATALOGO:
            criterio, base = CATALOGO[modo]
            texto = "Criterio del manuscrito: "+criterio+". Usa enteros, fracciones, I, sqrt o zeta(n,r)."
            if modo in "MOPWZ":
                texto += " El cálculo toma s = sqrt(f)."
        else:
            _, base, texto = MODELOS[modo]
        ayuda.value = "<p>"+html.escape(texto)+"</p>"
        campos.clear()
        for k, v in base.items():
            clase = widgets.Textarea if k == "matriz" else widgets.Text
            campo = clase(description=k, value=v, layout=widgets.Layout(width="100%"))
            campo.observe(marcar_pendiente, names="value")
            campos[k] = campo
        formulario.children = tuple(campos.values())
        estado.value = ""
        marcar_pendiente()

    def ejecutar(_=None):
        nonlocal actual
        modo, params = selector.value, valores()
        try:
            info = evaluar_panel(modo, params)
        except (SyntaxError, ValueError, TypeError, ArithmeticError, sp.polys.polyerrors.PolynomialError) as error:
            resultado.value = "<p><strong>Revisa los parámetros.</strong> "+html.escape(str(error))+"</p>"
            actual = None
            agregar.disabled = True
            return
        resultado.value = fila_html(info)
        actual = (modo, params, info)
        agregar.disabled = False
        estado.value = ""

    def guardar(_=None):
        if actual is None:
            return
        modo, params, info = actual
        nombre = "Familia "+modo if modo in CATALOGO else MODELOS[modo][0]
        pi = info.get("PI")
        if pi is None and "Grado PI" in info:
            pi = True  # Estos módulos se parametrizan por raíces de la unidad.
        fila = {"Álgebra": nombre, "Parámetros": ", ".join(f"{k}={v}" for k,v in params.items()),
                "PI": "Sí" if pi is True else "No" if pi is False else "—",
                "Grado PI": str(info["Grado PI"]) if info.get("Grado PI") is not None else "—",
                "Ozono": info.get("Grupo de ozono",info.get("Ozono",info.get("Orden de Oz","—"))),
                "Estado": info.get("Estado",info.get("Estado de Oz","—"))}
        if fila not in guardados:
            guardados.append(fila)
            estado.value = "<p>Caso añadido al comparador.</p>"
        else:
            estado.value = "<p>El caso ya está en el comparador.</p>"
        actualizar_tabla()

    def exportar(_=None):
        if not guardados:
            return
        destino = Path("comparacion_pbw.csv")
        with destino.open("w", encoding="utf-8-sig", newline="") as archivo:
            escritor = DictWriter(archivo, fieldnames=list(guardados[0]))
            escritor.writeheader()
            escritor.writerows(guardados)
        try:
            from google.colab import files
        except ImportError:
            enlaces.outputs = ()
            enlaces.append_display_data(FileLink(destino))
        else:
            files.download(str(destino))

    def vaciar(_=None):
        guardados.clear()
        enlaces.outputs = ()
        actualizar_tabla()

    selector.observe(configurar, names="value")
    calcular.on_click(ejecutar)
    agregar.on_click(guardar)
    descargar.on_click(exportar)
    limpiar.on_click(vaciar)
    configurar()
    actualizar_tabla()
    controles = widgets.Box((calcular, agregar), layout=widgets.Layout(display="flex", flex_flow="row wrap", gap="6px"))
    controles_tabla = widgets.Box((descargar, limpiar), layout=widgets.Layout(display="flex", flex_flow="row wrap", gap="6px"))
    panel = widgets.VBox((widgets.HTML("<h3>Calculadora PBW</h3>"), selector, ayuda, formulario,
                          controles, resultado, estado, widgets.HTML("<h3>Comparador</h3>"),
                          controles_tabla, comparador, enlaces), layout=ancho)
    if mostrar:
        display(panel)
    return panel

panel_pbw = abrir_panel()
